In [4]:
import os
import time
import pandas as pd
from moexalgo import Ticker

# 1. Настройка параметров выгрузки
TICKERS = ['SBER', 'GAZP', 'VTBR', 'LKOH','TCSG', 'ROSN', 'NVTK', 'OZON', 'PLZL', 'GMKN',
         'SELG', 'SMLT', 'YDEX', 'X5', 'RUAL', 'MGNT', 'ALRS', 'MTSS', 'IRAO', 'POSI' ]
START_DATE = '2015-01-01'
END_DATE = '2026-10-06'

# 2. Создание структуры папок
output_dir = os.path.join('data', 'raw')
os.makedirs(output_dir, exist_ok=True)

parquet_path = os.path.join(output_dir, 'moex_stocks_panel.parquet')
csv_path = os.path.join(output_dir, 'moex_stocks_panel.csv')

all_data = []

print("Начало пакетной выгрузки данных с MOEX...")

# 3. Цикл загрузки данных по инструментам
for ticker_symbol in TICKERS:
    print(f"Загрузка котировок: {ticker_symbol}...")
    try:
        ticker = Ticker(ticker_symbol)
        # В moexalgo дневной таймфрейм задается как '1D'
        candles_iter = ticker.candles(start=START_DATE, end=END_DATE, period='1D')
        df_ticker = pd.DataFrame(candles_iter)
        
        if df_ticker.empty:
            print(f"  Внимание: по тикеру {ticker_symbol} данные не найдены.")
            continue
            
        # Добавляем идентификатор тикера и приводим дату
        df_ticker['ticker'] = ticker_symbol
        df_ticker['begin'] = pd.to_datetime(df_ticker['begin'])
        
        # Оставляем только базовые столбцы ценового бара
        columns_to_keep = ['begin', 'ticker', 'open', 'close', 'high', 'low', 'value', 'volume']
        df_ticker = df_ticker[[col for col in columns_to_keep if col in df_ticker.columns]]
        
        all_data.append(df_ticker)
        time.sleep(0.3)  # пауза между запросами для предотвращения сетевых ограничений
        
    except Exception as e:
        print(f"  Ошибка при обработке {ticker_symbol}: {e}")

# 4. Формирование единой панели данных
if not all_data:
    raise RuntimeError("Не удалось загрузить данные ни по одному тикеру.")

panel_df = pd.concat(all_data, ignore_index=True)

# Сортировка по тикеру и хронологии
panel_df.sort_values(by=['ticker', 'begin'], inplace=True)
panel_df.reset_index(drop=True, inplace=True)

# 5. Сохранение в data/raw/
#panel_df.to_parquet(parquet_path, index=False)
panel_df.to_csv(csv_path, index=False)

print("\n--- Выгрузка успешно завершена ---")
print(f"Итоговый размер таблицы: {panel_df.shape[0]} строк, {panel_df.shape[1]} столбцов")
print(f"Диапазон дат: с {panel_df['begin'].min().date()} по {panel_df['begin'].max().date()}")
print(f"Файлы сохранены:\n  • {parquet_path}\n  • {csv_path}")

print("\nПример первых 5 строк:")
print(panel_df.head())

Начало пакетной выгрузки данных с MOEX...
Загрузка котировок: SBER...
Загрузка котировок: GAZP...
Загрузка котировок: VTBR...
Загрузка котировок: LKOH...
Загрузка котировок: TCSG...
Загрузка котировок: ROSN...
Загрузка котировок: NVTK...
Загрузка котировок: OZON...
Загрузка котировок: PLZL...
Загрузка котировок: GMKN...
Загрузка котировок: SELG...
Загрузка котировок: SMLT...
Загрузка котировок: YDEX...
Загрузка котировок: X5...
Загрузка котировок: RUAL...
Загрузка котировок: MGNT...
Загрузка котировок: ALRS...
Загрузка котировок: MTSS...
Загрузка котировок: IRAO...
Загрузка котировок: POSI...

--- Выгрузка успешно завершена ---
Итоговый размер таблицы: 50016 строк, 8 столбцов
Диапазон дат: с 2015-01-05 по 2026-10-06
Файлы сохранены:
  • data\raw\moex_stocks_panel.parquet
  • data\raw\moex_stocks_panel.csv

Пример первых 5 строк:
       begin ticker   open  close   high    low        value   volume
0 2015-01-05   ALRS  63.95  60.38  64.49  58.91  255440582.0  4150300
1 2015-01-06   ALRS

In [5]:
panel_df

,begin,ticker,open,close,high,low,value,volume
0,2015-01-05,ALRS,63.95,60.38,64.49,58.91,2.554406e+08,4150300
1,2015-01-06,ALRS,60.20,61.28,62.59,59.89,1.579549e+08,2560000
2,2015-01-08,ALRS,61.28,60.20,63.55,57.54,5.483854e+08,8930000
3,2015-01-09,ALRS,59.51,61.91,63.00,59.51,2.814642e+08,4551600
4,2015-01-12,ALRS,62.94,63.00,63.50,61.10,1.596798e+08,2552000
...,...,...,...,...,...,...,...,...
50011,2026-10-02,YDEX,3608.00,3603.00,3620.50,3576.00,9.800233e+08,272489
50012,2026-10-03,YDEX,3608.00,3604.50,3613.50,3602.00,7.084714e+07,19630
50013,2026-10-04,YDEX,3604.50,3600.50,3604.50,3584.50,1.189503e+08,33103
50014,2026-10-05,YDEX,3601.50,3675.00,3690.00,3588.00,2.849881e+09,778618
